# Multi-Agent E-commerce Dispute Resolution — Qwen3-8B

Notebook này xử lý 50 case Olist theo `EC_POLICY_V1` bằng các agent có quyền truy cập và handoff tách biệt:

- **Order & Seller Agent**: trạng thái order, items, sellers, shipping limits và item/freight totals.
- **Payment Agent**: payment rows, payment total và reconciliation.
- **Delivery Agent**: đối chiếu actual/estimated delivery và seller handoff deadline.
- **Policy Agent**: dùng `Qwen/Qwen3-8B` để đề xuất issue/cause/party từ ba handoff.
- **Verifier Agent**: kiểm tra lại bằng policy engine xác định, schema, tiền và evidence allowlist.
- **Coordinator Agent**: điều phối, ghi output, trace và metadata.

Qwen không được tạo số tiền hoặc evidence ID. Nếu dependency/model/GPU/inference/JSON lỗi, hoặc đề xuất của model mâu thuẫn với facts, hệ thống tự động dùng deterministic fallback và vẫn sinh đủ 50 JSON hợp lệ.

**Kaggle**: bật GPU, attach repo/dataset chứa `data/` + `input/`; nên attach model Qwen3-8B vào `/kaggle/input` để chạy được cả khi Internet tắt. Kết quả nằm tại `/kaggle/working/output`, `/kaggle/working/submission.zip`, `trace.jsonl` và `metadata.json`.


In [ ]:
# Optional dependency bootstrap. Failure is non-fatal because the rule fallback has no LLM dependency.
from pathlib import Path
import importlib.metadata as importlib_metadata
import os
import re
import subprocess
import sys

IS_KAGGLE = Path("/kaggle").exists()


def _version_tuple(value):
    numbers = re.findall(r"\d+", str(value))
    return tuple(int(part) for part in (numbers + ["0", "0", "0"])[:3])


def _installed_version(package):
    try:
        return importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        return None


install_requested = os.getenv("EC_INSTALL_DEPS", "1" if IS_KAGGLE else "0") == "1"
transformers_version = _installed_version("transformers")
bitsandbytes_version = _installed_version("bitsandbytes")
needs_install = (
    transformers_version is None
    or _version_tuple(transformers_version) < (4, 51, 0)
    or bitsandbytes_version is None
)

if install_requested and needs_install:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "transformers>=4.51.0,<5",
        "accelerate>=1.2.0",
        "bitsandbytes>=0.43.0",
    ]
    try:
        completed = subprocess.run(command, capture_output=True, text=True, timeout=360, check=False)
        if completed.returncode:
            print("Dependency install failed; deterministic fallback remains available.")
            print((completed.stderr or completed.stdout)[-1200:])
        else:
            print("Qwen dependencies are ready.")
    except Exception as exc:
        print(f"Dependency install skipped after error: {type(exc).__name__}: {exc}")
else:
    print(
        "Dependency bootstrap not needed/requested:",
        {"transformers": transformers_version, "bitsandbytes": bitsandbytes_version},
    )


In [ ]:
from collections import Counter
from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
import gc
import json
import math
import platform
import time
import uuid
import zipfile

import pandas as pd

MODEL_ID = "Qwen/Qwen3-8B"  # Required model name is intentionally declared in source.
MODEL_PARAMETER_SIZE = "8.2B"
POLICY_VERSION = "EC_POLICY_V1"
EXPECTED_CASE_COUNT = 50
MONEY_QUANTUM = Decimal("0.01")
RECONCILIATION_TOLERANCE = Decimal("0.10")
VERIFIED_CONFIDENCE = 0.99
MAX_NEW_TOKENS = int(os.getenv("QWEN_MAX_NEW_TOKENS", "96"))


def env_flag(name, default):
    return os.getenv(name, "1" if default else "0").strip().lower() in {
        "1",
        "true",
        "yes",
        "on",
    }


ENABLE_LLM = env_flag("EC_ENABLE_LLM", True) and not env_flag("EC_FORCE_RULE_FALLBACK", False)
ALLOW_MODEL_DOWNLOAD = env_flag("QWEN_ALLOW_DOWNLOAD", True)
MIRROR_LOGGING = env_flag("EC_MIRROR_LOGGING", True)
STRICT_OFFICIAL_ASSERTIONS = env_flag("EC_STRICT_OFFICIAL_ASSERTIONS", True)


def resolve_marker_dir(env_name, marker_name, local_subdir):
    explicit = os.getenv(env_name)
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if (candidate / marker_name).is_file():
            return candidate
        raise FileNotFoundError(f"{env_name}={candidate} does not contain {marker_name}")

    preferred = [Path.cwd() / local_subdir, Path.cwd()]
    for candidate in preferred:
        if (candidate / marker_name).is_file():
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob(marker_name), key=lambda p: (len(p.parts), str(p)))
        if matches:
            return matches[0].parent.resolve()
    raise FileNotFoundError(
        f"Cannot locate {marker_name}. Set {env_name} or attach the project dataset on Kaggle."
    )


DATA_DIR = resolve_marker_dir("EC_DATA_DIR", "olist_orders_dataset.csv", "data")
INPUT_DIR = resolve_marker_dir("EC_INPUT_DIR", "EC_001.json", "input")
default_work_root = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
WORK_ROOT = Path(os.getenv("EC_WORK_ROOT", str(default_work_root))).expanduser().resolve()
OUTPUT_DIR = WORK_ROOT / "output"
LOGGING_DIR = WORK_ROOT / "logging"
ROOT_TRACE_PATH = WORK_ROOT / "trace.jsonl"
ROOT_METADATA_PATH = WORK_ROOT / "metadata.json"
LOG_TRACE_PATH = LOGGING_DIR / "trace.jsonl"
LOG_METADATA_PATH = LOGGING_DIR / "metadata.json"
SUBMISSION_ZIP = WORK_ROOT / "submission.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGGING_DIR.mkdir(parents=True, exist_ok=True)

print(
    json.dumps(
        {
            "environment": "kaggle" if IS_KAGGLE else "local",
            "data_dir": str(DATA_DIR),
            "input_dir": str(INPUT_DIR),
            "work_root": str(WORK_ROOT),
            "model": MODEL_ID,
            "llm_requested": ENABLE_LLM,
            "allow_model_download": ALLOW_MODEL_DOWNLOAD,
        },
        indent=2,
    )
)


In [ ]:
# Load only the four tables needed by EC_POLICY_V1. Items and payments stay separate to avoid
# many-to-many multiplication of monetary values.
REQUIRED_COLUMNS = {
    "orders": {
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    },
    "items": {
        "order_id",
        "order_item_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value",
    },
    "payments": {
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value",
    },
    "sellers": {"seller_id"},
}


def read_csv_strings(filename):
    return pd.read_csv(DATA_DIR / filename, dtype=str, keep_default_na=False)


orders_df = read_csv_strings("olist_orders_dataset.csv")
items_df = read_csv_strings("olist_order_items_dataset.csv")
payments_df = read_csv_strings("olist_order_payments_dataset.csv")
sellers_df = read_csv_strings("olist_sellers_dataset.csv")

for name, frame in {
    "orders": orders_df,
    "items": items_df,
    "payments": payments_df,
    "sellers": sellers_df,
}.items():
    missing = REQUIRED_COLUMNS[name] - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

if orders_df["order_id"].duplicated().any():
    raise ValueError("orders.order_id must be unique")
if items_df[["order_id", "order_item_id"]].duplicated().any():
    raise ValueError("(order_id, order_item_id) must be unique")
if payments_df[["order_id", "payment_sequential"]].duplicated().any():
    raise ValueError("(order_id, payment_sequential) must be unique")


def money_decimal(value):
    if isinstance(value, Decimal):
        number = value
    else:
        text = "0" if value is None else str(value).strip()
        if text == "" or text.lower() in {"nan", "nat", "none"}:
            text = "0"
        try:
            number = Decimal(text)
        except InvalidOperation as exc:
            raise ValueError(f"Invalid monetary value: {value!r}") from exc
    return number.quantize(MONEY_QUANTUM, rounding=ROUND_HALF_UP)


def sum_money(values):
    total = sum((Decimal(str(value).strip() or "0") for value in values), Decimal("0"))
    return money_decimal(total)


def money_float(value):
    return float(money_decimal(value))


def parsed_timestamp(value):
    if value is None or not str(value).strip():
        return None
    result = pd.to_datetime(str(value), errors="coerce")
    return None if pd.isna(result) else result


def numeric_sequence(value):
    try:
        return int(str(value))
    except ValueError:
        return str(value)


source_row_counts = {
    "orders": len(orders_df),
    "items": len(items_df),
    "payments": len(payments_df),
}
known_order_ids = set(orders_df["order_id"])
known_seller_ids = set(sellers_df["seller_id"])

case_files = sorted(INPUT_DIR.glob("EC_*.json"))
expected_input_names = {f"EC_{index:03d}.json" for index in range(1, EXPECTED_CASE_COUNT + 1)}
if {path.name for path in case_files} != expected_input_names:
    missing = sorted(expected_input_names - {path.name for path in case_files})
    extra = sorted({path.name for path in case_files} - expected_input_names)
    raise ValueError(f"Input set mismatch. Missing={missing}, extra={extra}")

CASES = []
seen_case_ids = set()
seen_order_ids = set()
for path in case_files:
    case = json.loads(path.read_text(encoding="utf-8"))
    case_id = case.get("case_id")
    order_id = case.get("customer_request", {}).get("claimed_order_id")
    if path.stem != case_id:
        raise ValueError(f"Filename/case_id mismatch: {path.name} vs {case_id}")
    if case.get("policy_version") != POLICY_VERSION:
        raise ValueError(f"Unsupported policy for {case_id}: {case.get('policy_version')}")
    if order_id not in known_order_ids:
        raise KeyError(f"Claimed order not found for {case_id}: {order_id}")
    if case_id in seen_case_ids or order_id in seen_order_ids:
        raise ValueError(f"Duplicate case/order detected at {case_id}")
    seen_case_ids.add(case_id)
    seen_order_ids.add(order_id)
    case["_source_filename"] = path.name
    CASES.append(case)

# Keep only the 50 target orders before grouping. This avoids ~200k tiny DataFrames in RAM.
target_order_ids = set(seen_order_ids)
orders_df = orders_df[orders_df["order_id"].isin(target_order_ids)].copy()
items_df = items_df[items_df["order_id"].isin(target_order_ids)].copy()
payments_df = payments_df[payments_df["order_id"].isin(target_order_ids)].copy()
order_lookup = {row["order_id"]: row.to_dict() for _, row in orders_df.iterrows()}
items_by_order = {
    order_id: group.sort_values("order_item_id", key=lambda s: s.map(numeric_sequence)).copy()
    for order_id, group in items_df.groupby("order_id", sort=False)
}
payments_by_order = {
    order_id: group.sort_values("payment_sequential", key=lambda s: s.map(numeric_sequence)).copy()
    for order_id, group in payments_df.groupby("order_id", sort=False)
}
del known_order_ids
gc.collect()

print(
    f"Loaded {len(CASES)} cases from source rows {source_row_counts}; retained "
    f"{len(orders_df)} orders, {len(items_df)} items and {len(payments_df)} payments."
)


In [ ]:
def json_safe(value):
    if isinstance(value, Decimal):
        return money_float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if hasattr(value, "item") and callable(value.item):
        return value.item()
    return value


def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


class TraceRecorder:
    def __init__(self):
        self.run_id = uuid.uuid4().hex
        self.started_at = datetime.now(timezone.utc)
        self.events = []

    def emit(self, case_id, agent, event, payload=None, from_agent=None, to_agent=None):
        record = {
            "run_id": self.run_id,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "case_id": case_id,
            "agent": agent,
            "event": event,
        }
        if from_agent is not None:
            record["from_agent"] = from_agent
        if to_agent is not None:
            record["to_agent"] = to_agent
        if payload is not None:
            record["payload"] = json_safe(payload)
        self.events.append(record)

    def flush(self, paths):
        content = "".join(
            json.dumps(event, ensure_ascii=False, allow_nan=False) + "\n" for event in self.events
        )
        for path in dict.fromkeys(Path(item) for item in paths):
            atomic_write_text(path, content)


def parse_first_json_object(text):
    cleaned = re.sub(r"<think>.*?</think>", "", str(text), flags=re.DOTALL | re.IGNORECASE)
    cleaned = cleaned.replace("```json", "").replace("```JSON", "").replace("```", "")
    decoder = json.JSONDecoder()
    for index, character in enumerate(cleaned):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(cleaned[index:])
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            continue
    raise ValueError("No JSON object found in model response")


class QwenGateway:
    """One shared Qwen instance. It can only propose policy fields from structured handoffs."""

    def __init__(self, model_id=MODEL_ID, enabled=ENABLE_LLM):
        self.model_id = model_id
        self.enabled = enabled
        self.ready = False
        self.status = "not_initialized"
        self.model_source = None
        self.quantization = None
        self.load_errors = []
        self.calls = 0
        self.responses_parsed = 0
        self.generation_errors = 0
        self.tokenizer = None
        self.model = None
        self.torch = None

    @staticmethod
    def _attached_model_path():
        explicit = os.getenv("QWEN_MODEL_PATH")
        if explicit:
            candidate = Path(explicit).expanduser()
            if (candidate / "config.json").is_file():
                return candidate.resolve()
        kaggle_input = Path("/kaggle/input")
        if not kaggle_input.exists():
            return None
        candidates = []
        for config_path in kaggle_input.rglob("config.json"):
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if config.get("model_type") != "qwen3":
                continue
            if config.get("hidden_size") == 4096 and config.get("num_hidden_layers") == 36:
                priority = 0 if "8b" in str(config_path).lower() else 1
                candidates.append((priority, len(config_path.parts), str(config_path), config_path.parent))
        return sorted(candidates)[0][-1].resolve() if candidates else None

    def _release_partial_model(self):
        self.model = None
        self.tokenizer = None
        gc.collect()
        if self.torch is not None and self.torch.cuda.is_available():
            self.torch.cuda.empty_cache()

    def load(self):
        if not self.enabled:
            self.status = "disabled_deterministic_fallback"
            return self
        try:
            import torch
            import transformers
            from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

            self.torch = torch
            if _version_tuple(transformers.__version__) < (4, 51, 0):
                raise RuntimeError(
                    f"transformers>={4.51} is required for qwen3; found {transformers.__version__}"
                )
            if not torch.cuda.is_available():
                raise RuntimeError("CUDA GPU unavailable; CPU load is intentionally skipped")
        except Exception as exc:
            self.status = "dependency_or_cuda_fallback"
            self.load_errors.append(f"{type(exc).__name__}: {exc}")
            return self

        attached = self._attached_model_path()
        sources = []
        if attached is not None:
            sources.append((str(attached), True, "attached_local"))
        sources.append((self.model_id, True, "huggingface_cache"))
        if ALLOW_MODEL_DOWNLOAD:
            sources.append((self.model_id, False, "huggingface_hub"))

        unique_sources = []
        seen = set()
        for source in sources:
            key = (source[0], source[1])
            if key not in seen:
                seen.add(key)
                unique_sources.append(source)

        compute_dtype = (
            self.torch.bfloat16
            if hasattr(self.torch.cuda, "is_bf16_supported") and self.torch.cuda.is_bf16_supported()
            else self.torch.float16
        )
        total_vram = sum(
            self.torch.cuda.get_device_properties(index).total_memory
            for index in range(self.torch.cuda.device_count())
        )

        for source, local_only, source_label in unique_sources:
            try:
                tokenizer = AutoTokenizer.from_pretrained(
                    source,
                    local_files_only=local_only,
                    trust_remote_code=False,
                )
                if tokenizer.pad_token_id is None:
                    tokenizer.pad_token_id = tokenizer.eos_token_id
                quantization_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_compute_dtype=compute_dtype,
                )
                model = AutoModelForCausalLM.from_pretrained(
                    source,
                    local_files_only=local_only,
                    trust_remote_code=False,
                    device_map={"": 0},
                    quantization_config=quantization_config,
                    low_cpu_mem_usage=True,
                )
                model.eval()
                self.tokenizer = tokenizer
                self.model = model
                self.model_source = source_label + ":" + source
                self.quantization = "bitsandbytes_nf4_4bit"
                self.status = "ready"
                self.ready = True
                return self
            except Exception as exc:
                self.load_errors.append(
                    f"{source_label}/nf4: {type(exc).__name__}: {str(exc)[:500]}"
                )
                self._release_partial_model()

            # Full precision is attempted only when aggregate VRAM is safely above raw FP16 weights.
            if total_vram >= 22 * 1024**3:
                try:
                    tokenizer = AutoTokenizer.from_pretrained(
                        source,
                        local_files_only=local_only,
                        trust_remote_code=False,
                    )
                    if tokenizer.pad_token_id is None:
                        tokenizer.pad_token_id = tokenizer.eos_token_id
                    model = AutoModelForCausalLM.from_pretrained(
                        source,
                        local_files_only=local_only,
                        trust_remote_code=False,
                        device_map="auto",
                        torch_dtype=compute_dtype,
                        low_cpu_mem_usage=True,
                    )
                    model.eval()
                    self.tokenizer = tokenizer
                    self.model = model
                    self.model_source = source_label + ":" + source
                    self.quantization = str(compute_dtype).replace("torch.", "") + "_sharded"
                    self.status = "ready"
                    self.ready = True
                    return self
                except Exception as exc:
                    self.load_errors.append(
                        f"{source_label}/full: {type(exc).__name__}: {str(exc)[:500]}"
                    )
                    self._release_partial_model()

        self.status = "model_load_fallback"
        return self

    @staticmethod
    def _input_device(model):
        device_map = getattr(model, "hf_device_map", None) or {}
        for value in device_map.values():
            text = str(value)
            if text.startswith("cuda") or isinstance(value, int):
                return value if isinstance(value, str) else f"cuda:{value}"
        return next(model.parameters()).device

    def propose_policy(self, handoffs):
        if not self.ready:
            return None, {"ok": False, "reason": self.status, "latency_ms": 0.0}

        system_prompt = """You are the Policy Agent for EC_POLICY_V1. Use only the JSON handoffs.
Apply this exact mapping in strict priority:
1. canceled+paid -> canceled_order_paid | ORDER_CANCELED_AFTER_PAYMENT | platform | OLIST_PLATFORM
2. unavailable+paid -> unavailable_order_paid | ORDER_UNAVAILABLE_AFTER_PAYMENT | platform | OLIST_PLATFORM
3. delivery late + seller handoff late -> late_delivery_seller | SELLER_HANDOFF_AFTER_LIMIT | seller | violating seller ID
4. delivery late + seller handoff on time -> late_delivery_logistics | CARRIER_DELIVERED_AFTER_ESTIMATE | logistics_provider | LOGISTICS_PROVIDER
5. at least two payments + reconciled -> valid_split_payment | MULTIPLE_PAYMENTS_RECONCILED | null | null
6. delivery within estimate + reconciled -> unsupported_late_claim | DELIVERY_WITHIN_ESTIMATE | null | null
Return exactly one compact JSON object with keys primary_issue, cause_code, party_type, party_id.
Use the exact tokens above. Never invent IDs or money."""
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(json_safe(handoffs), ensure_ascii=False)},
        ]
        started = time.perf_counter()
        self.calls += 1
        try:
            try:
                prompt = self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            except TypeError:
                messages[0]["content"] += " /no_think"
                prompt = self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            tokenized = self.tokenizer(prompt, return_tensors="pt")
            input_device = self._input_device(self.model)
            tokenized = {key: value.to(input_device) for key, value in tokenized.items()}
            with self.torch.inference_mode():
                generated = self.model.generate(
                    **tokenized,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    use_cache=True,
                    eos_token_id=self.tokenizer.eos_token_id,
                    pad_token_id=self.tokenizer.pad_token_id,
                )
            new_tokens = generated[0][tokenized["input_ids"].shape[-1] :]
            raw_text = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            candidate = parse_first_json_object(raw_text)
            self.responses_parsed += 1
            return candidate, {
                "ok": True,
                "reason": None,
                "latency_ms": round((time.perf_counter() - started) * 1000, 2),
                "raw_response": raw_text[:500],
            }
        except Exception as exc:
            self.generation_errors += 1
            reason = f"{type(exc).__name__}: {str(exc)[:500]}"
            if self.torch is not None and (
                "out of memory" in reason.lower()
                or isinstance(exc, getattr(self.torch.cuda, "OutOfMemoryError", RuntimeError))
            ):
                self.ready = False
                self.status = "generation_oom_fallback"
                self._release_partial_model()
            return None, {
                "ok": False,
                "reason": reason,
                "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            }


# A second Run All in the same Kaggle kernel must release the previous 8B model first.
_previous_coordinator = globals().get("COORDINATOR")
_previous_llm = globals().get("LLM")
for _previous_gateway in [
    getattr(getattr(_previous_coordinator, "policy", None), "gateway", None),
    _previous_llm,
]:
    if _previous_gateway is not None and hasattr(_previous_gateway, "_release_partial_model"):
        try:
            _previous_gateway._release_partial_model()
        except Exception:
            pass
if "COORDINATOR" in globals():
    del COORDINATOR
if "LLM" in globals():
    del LLM
gc.collect()

TRACE = TraceRecorder()
LLM = QwenGateway().load()
TRACE.emit(
    None,
    "QwenGateway",
    "backend_initialized",
    {
        "configured_model": MODEL_ID,
        "status": LLM.status,
        "model_source": LLM.model_source,
        "quantization": LLM.quantization,
        "load_errors": LLM.load_errors,
    },
)
print(
    json.dumps(
        {
            "qwen_status": LLM.status,
            "model_source": LLM.model_source,
            "quantization": LLM.quantization,
            "load_error_count": len(LLM.load_errors),
        },
        indent=2,
    )
)


In [ ]:
ISSUE_SPECS = {
    "canceled_order_paid": {
        "cause_code": "ORDER_CANCELED_AFTER_PAYMENT",
        "action": "issue_full_refund",
        "party_type": "platform",
        "party_id": "OLIST_PLATFORM",
    },
    "unavailable_order_paid": {
        "cause_code": "ORDER_UNAVAILABLE_AFTER_PAYMENT",
        "action": "issue_full_refund",
        "party_type": "platform",
        "party_id": "OLIST_PLATFORM",
    },
    "late_delivery_seller": {
        "cause_code": "SELLER_HANDOFF_AFTER_LIMIT",
        "action": "refund_freight",
        "party_type": "seller",
        "party_id": None,
    },
    "late_delivery_logistics": {
        "cause_code": "CARRIER_DELIVERED_AFTER_ESTIMATE",
        "action": "refund_freight",
        "party_type": "logistics_provider",
        "party_id": "LOGISTICS_PROVIDER",
    },
    "valid_split_payment": {
        "cause_code": "MULTIPLE_PAYMENTS_RECONCILED",
        "action": "explain_valid_split_payment",
        "party_type": None,
        "party_id": None,
    },
    "unsupported_late_claim": {
        "cause_code": "DELIVERY_WITHIN_ESTIMATE",
        "action": "reject_late_refund",
        "party_type": None,
        "party_id": None,
    },
}


class OrderSellerAgent:
    def analyze(self, case):
        case_id = case["case_id"]
        order_id = case["customer_request"]["claimed_order_id"]
        order = order_lookup[order_id]
        item_rows = items_by_order.get(order_id, items_df.iloc[0:0])
        item_records = []
        for _, row in item_rows.iterrows():
            seller_id = row["seller_id"]
            if seller_id not in known_seller_ids:
                raise ValueError(f"Unknown seller {seller_id} for {case_id}")
            item_records.append(
                {
                    "item_id": f"{order_id}:{row['order_item_id']}",
                    "order_item_id": str(row["order_item_id"]),
                    "seller_id": seller_id,
                    "shipping_limit_date": row["shipping_limit_date"],
                    "price_brl": money_decimal(row["price"]),
                    "freight_brl": money_decimal(row["freight_value"]),
                }
            )
        handoff = {
            "order_id": order_id,
            "order_status": order["order_status"],
            "order_delivered_carrier_date": order["order_delivered_carrier_date"],
            "order_delivered_customer_date": order["order_delivered_customer_date"],
            "order_estimated_delivery_date": order["order_estimated_delivery_date"],
            "items": item_records,
            "seller_ids": list(dict.fromkeys(item["seller_id"] for item in item_records)),
            "item_total_brl": sum_money(item["price_brl"] for item in item_records),
            "freight_total_brl": sum_money(item["freight_brl"] for item in item_records),
        }
        TRACE.emit(
            case_id,
            "OrderSellerAgent",
            "handoff",
            handoff,
            from_agent="OrderSellerAgent",
            to_agent="CoordinatorAgent",
        )
        return handoff


class PaymentAgent:
    def analyze(self, case, financial_scope):
        case_id = case["case_id"]
        order_id = financial_scope["order_id"]
        payment_rows = payments_by_order.get(order_id, payments_df.iloc[0:0])
        records = []
        for _, row in payment_rows.iterrows():
            records.append(
                {
                    "payment_id": f"{order_id}:{row['payment_sequential']}",
                    "payment_sequential": str(row["payment_sequential"]),
                    "payment_type": row["payment_type"],
                    "payment_value_brl": money_decimal(row["payment_value"]),
                }
            )
        payment_total = sum_money(record["payment_value_brl"] for record in records)
        expected_total = money_decimal(
            financial_scope["item_total_brl"] + financial_scope["freight_total_brl"]
        )
        difference = money_decimal(abs(payment_total - expected_total))
        handoff = {
            "order_id": order_id,
            "payments": records,
            "payment_count": len(records),
            "payment_total_brl": payment_total,
            "expected_item_plus_freight_brl": expected_total,
            "difference_brl": difference,
            "reconciled": difference <= RECONCILIATION_TOLERANCE,
        }
        TRACE.emit(
            case_id,
            "PaymentAgent",
            "handoff",
            handoff,
            from_agent="PaymentAgent",
            to_agent="CoordinatorAgent",
        )
        return handoff


class DeliveryAgent:
    def analyze(self, case, delivery_scope):
        case_id = case["case_id"]
        delivered = parsed_timestamp(delivery_scope["order_delivered_customer_date"])
        estimated = parsed_timestamp(delivery_scope["order_estimated_delivery_date"])
        carrier = parsed_timestamp(delivery_scope["order_delivered_carrier_date"])
        late_delivery = None if delivered is None or estimated is None else delivered > estimated
        violating_sellers = []
        shipping_limits = []
        if carrier is not None:
            for item in delivery_scope["items"]:
                shipping_limit = parsed_timestamp(item["shipping_limit_date"])
                shipping_limits.append(shipping_limit)
                if shipping_limit is not None and carrier > shipping_limit:
                    violating_sellers.append(item["seller_id"])
        else:
            shipping_limits = [
                parsed_timestamp(item["shipping_limit_date"])
                for item in delivery_scope["items"]
            ]
        shipping_limits_complete = bool(delivery_scope["items"]) and all(
            limit is not None for limit in shipping_limits
        )
        carrier_handoff_verified_on_time = (
            carrier is not None
            and shipping_limits_complete
            and all(carrier <= limit for limit in shipping_limits)
        )
        handoff = {
            "order_id": delivery_scope["order_id"],
            "late_delivery": late_delivery,
            "seller_handoff_after_limit_ids": list(dict.fromkeys(violating_sellers)),
            "timestamps_complete_for_delivery": delivered is not None and estimated is not None,
            "carrier_timestamp_available": carrier is not None,
            "shipping_limits_complete": shipping_limits_complete,
            "carrier_handoff_verified_on_time": carrier_handoff_verified_on_time,
        }
        TRACE.emit(
            case_id,
            "DeliveryAgent",
            "handoff",
            handoff,
            from_agent="DeliveryAgent",
            to_agent="CoordinatorAgent",
        )
        return handoff


class PolicyCoverageError(RuntimeError):
    pass


class PolicyAgent:
    def __init__(self, gateway):
        self.gateway = gateway

    @staticmethod
    def deterministic_decision(order_handoff, payment_handoff, delivery_handoff):
        status = order_handoff["order_status"]
        payment_total = payment_handoff["payment_total_brl"]
        late_delivery = delivery_handoff["late_delivery"]
        late_sellers = delivery_handoff["seller_handoff_after_limit_ids"]

        if status == "canceled" and payment_total > 0:
            issue = "canceled_order_paid"
            refund = payment_total
            parties = [{"party_type": "platform", "party_id": "OLIST_PLATFORM"}]
        elif status == "unavailable" and payment_total > 0:
            issue = "unavailable_order_paid"
            refund = payment_total
            parties = [{"party_type": "platform", "party_id": "OLIST_PLATFORM"}]
        elif late_delivery is True and late_sellers:
            issue = "late_delivery_seller"
            refund = order_handoff["freight_total_brl"]
            parties = [
                {"party_type": "seller", "party_id": seller_id}
                for seller_id in late_sellers[:3]
            ]
        elif late_delivery is True and delivery_handoff["carrier_handoff_verified_on_time"]:
            issue = "late_delivery_logistics"
            refund = order_handoff["freight_total_brl"]
            parties = [
                {"party_type": "logistics_provider", "party_id": "LOGISTICS_PROVIDER"}
            ]
        elif payment_handoff["payment_count"] >= 2 and payment_handoff["reconciled"]:
            issue = "valid_split_payment"
            refund = Decimal("0")
            parties = []
        elif late_delivery is False and payment_handoff["reconciled"]:
            issue = "unsupported_late_claim"
            refund = Decimal("0")
            parties = []
        else:
            raise PolicyCoverageError(
                "Case is outside EC_POLICY_V1 coverage; refusing to invent a resolution"
            )

        spec = ISSUE_SPECS[issue]
        return {
            "primary_issue": issue,
            "cause_code": spec["cause_code"],
            "responsible_parties": parties,
            "recommended_refund_brl": money_decimal(refund),
            "resolution_action": spec["action"],
            "case_status": "action_required" if money_decimal(refund) > 0 else "no_action",
        }

    @staticmethod
    def _candidate_matches(candidate, expected):
        required_keys = {"primary_issue", "cause_code", "party_type", "party_id"}
        if not isinstance(candidate, dict) or set(candidate) != required_keys:
            return False
        if candidate["primary_issue"] != expected["primary_issue"]:
            return False
        if candidate["cause_code"] != expected["cause_code"]:
            return False
        parties = expected["responsible_parties"]
        if not parties:
            return candidate["party_type"] is None and candidate["party_id"] is None
        if len(parties) != 1:
            return False
        return (
            candidate["party_type"] == parties[0]["party_type"]
            and candidate["party_id"] == parties[0]["party_id"]
        )

    def decide(self, case, order_handoff, payment_handoff, delivery_handoff):
        expected = self.deterministic_decision(
            order_handoff, payment_handoff, delivery_handoff
        )
        llm_input = {
            "policy_version": POLICY_VERSION,
            "order": {"status": order_handoff["order_status"]},
            "payment": {
                "row_count": payment_handoff["payment_count"],
                "payment_total_brl": payment_handoff["payment_total_brl"],
                "reconciled": payment_handoff["reconciled"],
            },
            "delivery": {
                "late_delivery": delivery_handoff["late_delivery"],
                "seller_handoff_after_limit_ids": delivery_handoff[
                    "seller_handoff_after_limit_ids"
                ],
                "carrier_handoff_verified_on_time": delivery_handoff[
                    "carrier_handoff_verified_on_time"
                ],
            },
        }
        candidate, model_meta = self.gateway.propose_policy(llm_input)
        accepted = self._candidate_matches(candidate, expected)
        if accepted:
            decision_source = "qwen_validated"
            fallback_reason = None
        else:
            decision_source = "deterministic_fallback"
            fallback_reason = (
                model_meta.get("reason")
                if candidate is None
                else "model_candidate_conflicted_with_deterministic_policy"
            )
        decision = dict(expected)
        decision["decision_source"] = decision_source
        decision["fallback_reason"] = fallback_reason
        TRACE.emit(
            case["case_id"],
            "PolicyAgent",
            "decision",
            {
                "llm_input": llm_input,
                "model_candidate": candidate,
                "model_meta": model_meta,
                "accepted": accepted,
                "authoritative_decision": decision,
            },
            from_agent="PolicyAgent",
            to_agent="CoordinatorAgent",
        )
        return decision


def assemble_output(case, order_handoff, payment_handoff, decision):
    order_id = order_handoff["order_id"]
    item_ids = [item["item_id"] for item in order_handoff["items"]][:5]
    seller_ids = order_handoff["seller_ids"][:5]
    payment_ids = [payment["payment_id"] for payment in payment_handoff["payments"]][:5]
    evidence_ids = [f"order:{order_id}"]
    evidence_ids.extend(f"item:{item_id}" for item_id in item_ids)
    evidence_ids.extend(f"payment:{payment_id}" for payment_id in payment_ids)
    evidence_ids.extend(f"seller:{seller_id}" for seller_id in seller_ids)
    policy_evidence = f"policy:{decision['cause_code']}"
    evidence_ids = evidence_ids[:9] + [policy_evidence]

    return {
        "case_id": case["case_id"],
        "assessment": {
            "primary_issue": decision["primary_issue"],
            "case_status": decision["case_status"],
            "confidence": VERIFIED_CONFIDENCE,
        },
        "affected_entities": {
            "order_ids": [order_id],
            "item_ids": item_ids,
            "seller_ids": seller_ids,
            "payment_ids": payment_ids,
        },
        "root_cause_analysis": {
            "ranked_causes": [{"cause_code": decision["cause_code"], "rank": 1}],
            "responsible_parties": decision["responsible_parties"],
        },
        "evidence_ids": evidence_ids,
        "financial_resolution": {
            "currency": "BRL",
            "item_total_brl": money_float(order_handoff["item_total_brl"]),
            "freight_total_brl": money_float(order_handoff["freight_total_brl"]),
            "payment_total_brl": money_float(payment_handoff["payment_total_brl"]),
            "recommended_refund_brl": money_float(decision["recommended_refund_brl"]),
        },
        "resolution_actions": [decision["resolution_action"]],
    }


class VerifierAgent:
    ROOT_KEYS = {
        "case_id",
        "assessment",
        "affected_entities",
        "root_cause_analysis",
        "evidence_ids",
        "financial_resolution",
        "resolution_actions",
    }

    @staticmethod
    def _allowed_evidence(order_handoff, payment_handoff, cause_code):
        order_id = order_handoff["order_id"]
        allowed = {f"order:{order_id}", f"policy:{cause_code}"}
        allowed.update(f"item:{item['item_id']}" for item in order_handoff["items"])
        allowed.update(
            f"payment:{payment['payment_id']}" for payment in payment_handoff["payments"]
        )
        allowed.update(f"seller:{seller_id}" for seller_id in order_handoff["seller_ids"])
        return allowed

    def validate(self, candidate, canonical, order_handoff, payment_handoff):
        errors = []
        if not isinstance(candidate, dict) or set(candidate) != self.ROOT_KEYS:
            errors.append("root_schema_keys")
            return errors
        try:
            json.dumps(candidate, ensure_ascii=False, allow_nan=False)
        except (TypeError, ValueError):
            errors.append("not_strict_json")
        if candidate != canonical:
            errors.append("canonical_business_mismatch")

        assessment = candidate.get("assessment", {})
        issue = assessment.get("primary_issue")
        if issue not in ISSUE_SPECS:
            errors.append("primary_issue_enum")
            return errors
        confidence = assessment.get("confidence")
        if isinstance(confidence, bool) or not isinstance(confidence, (int, float)):
            errors.append("confidence_type")
        elif not 0 <= confidence <= 1:
            errors.append("confidence_range")

        entities = candidate.get("affected_entities", {})
        for key in ("order_ids", "item_ids", "seller_ids", "payment_ids"):
            if not isinstance(entities.get(key), list) or len(entities.get(key, [])) > 5:
                errors.append(f"entity_limit:{key}")
        roots = candidate.get("root_cause_analysis", {})
        if len(roots.get("ranked_causes", [])) > 3:
            errors.append("root_cause_limit")
        if len(roots.get("responsible_parties", [])) > 3:
            errors.append("responsible_party_limit")
        if len(candidate.get("resolution_actions", [])) > 5:
            errors.append("action_limit")
        if len(candidate.get("evidence_ids", [])) > 10:
            errors.append("evidence_limit")

        cause_code = ISSUE_SPECS[issue]["cause_code"]
        ranked = roots.get("ranked_causes", [])
        if ranked != [{"cause_code": cause_code, "rank": 1}]:
            errors.append("cause_or_rank_mismatch")
        if candidate.get("resolution_actions") != [ISSUE_SPECS[issue]["action"]]:
            errors.append("action_mismatch")

        allowed_evidence = self._allowed_evidence(
            order_handoff, payment_handoff, cause_code
        )
        if any(evidence not in allowed_evidence for evidence in candidate.get("evidence_ids", [])):
            errors.append("evidence_false_positive")

        refund = money_decimal(
            candidate.get("financial_resolution", {}).get("recommended_refund_brl", 0)
        )
        expected_status = "action_required" if refund > 0 else "no_action"
        if assessment.get("case_status") != expected_status:
            errors.append("status_refund_mismatch")
        return errors

    def verify_or_fallback(self, case, candidate, canonical, order_handoff, payment_handoff):
        errors = self.validate(candidate, canonical, order_handoff, payment_handoff)
        repaired = bool(errors)
        result = deepcopy(canonical) if repaired else candidate
        final_errors = self.validate(result, canonical, order_handoff, payment_handoff)
        if final_errors:
            raise ValueError(
                f"Canonical verifier failure for {case['case_id']}: {final_errors}"
            )
        TRACE.emit(
            case["case_id"],
            "VerifierAgent",
            "verification",
            {"valid": True, "repaired": repaired, "candidate_errors": errors},
            from_agent="VerifierAgent",
            to_agent="CoordinatorAgent",
        )
        return result, repaired


class CoordinatorAgent:
    def __init__(self, gateway):
        self.order_seller = OrderSellerAgent()
        self.payment = PaymentAgent()
        self.delivery = DeliveryAgent()
        self.policy = PolicyAgent(gateway)
        self.verifier = VerifierAgent()

    def process(self, case):
        started = time.perf_counter()
        case_id = case["case_id"]
        TRACE.emit(
            case_id,
            "CoordinatorAgent",
            "case_started",
            {
                "source_file": case["_source_filename"],
                "claimed_order_id": case["customer_request"]["claimed_order_id"],
                "policy_version": case["policy_version"],
            },
        )
        TRACE.emit(
            case_id,
            "CoordinatorAgent",
            "dispatch",
            {"scope": ["case_id", "claimed_order_id"]},
            from_agent="CoordinatorAgent",
            to_agent="OrderSellerAgent",
        )
        order_handoff = self.order_seller.analyze(case)
        financial_scope = {
            key: order_handoff[key]
            for key in ("order_id", "item_total_brl", "freight_total_brl")
        }
        TRACE.emit(
            case_id,
            "CoordinatorAgent",
            "dispatch",
            {"scope": sorted(financial_scope)},
            from_agent="CoordinatorAgent",
            to_agent="PaymentAgent",
        )
        payment_handoff = self.payment.analyze(case, financial_scope)
        delivery_scope = {
            "order_id": order_handoff["order_id"],
            "order_delivered_carrier_date": order_handoff["order_delivered_carrier_date"],
            "order_delivered_customer_date": order_handoff["order_delivered_customer_date"],
            "order_estimated_delivery_date": order_handoff["order_estimated_delivery_date"],
            "items": [
                {
                    "seller_id": item["seller_id"],
                    "shipping_limit_date": item["shipping_limit_date"],
                }
                for item in order_handoff["items"]
            ],
        }
        TRACE.emit(
            case_id,
            "CoordinatorAgent",
            "dispatch",
            {"scope": sorted(delivery_scope), "item_count": len(delivery_scope["items"])},
            from_agent="OrderSellerAgent",
            to_agent="DeliveryAgent",
        )
        delivery_handoff = self.delivery.analyze(case, delivery_scope)
        decision = self.policy.decide(
            case, order_handoff, payment_handoff, delivery_handoff
        )
        canonical = assemble_output(case, order_handoff, payment_handoff, decision)
        verified, verifier_repaired = self.verifier.verify_or_fallback(
            case, deepcopy(canonical), canonical, order_handoff, payment_handoff
        )
        elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
        run_info = {
            "decision_source": decision["decision_source"],
            "fallback_reason": decision["fallback_reason"],
            "verifier_repaired": verifier_repaired,
            "elapsed_ms": elapsed_ms,
        }
        TRACE.emit(
            case_id,
            "CoordinatorAgent",
            "case_completed",
            {
                **run_info,
                "primary_issue": verified["assessment"]["primary_issue"],
                "recommended_refund_brl": verified["financial_resolution"][
                    "recommended_refund_brl"
                ],
            },
        )
        return verified, run_info


In [ ]:
# Run all 50 cases. Existing EC_*.json files are replaced; unrelated files are untouched.
RUN_STARTED = time.perf_counter()
COORDINATOR = CoordinatorAgent(LLM)
RESULTS = {}
RUN_INFO = {}

for stale_output in OUTPUT_DIR.glob("EC_*.json"):
    stale_output.unlink()

for index, case in enumerate(CASES, start=1):
    result, run_info = COORDINATOR.process(case)
    output_path = OUTPUT_DIR / case["_source_filename"]
    atomic_write_text(
        output_path,
        json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    )
    RESULTS[case["case_id"]] = result
    RUN_INFO[case["case_id"]] = run_info
    if index % 10 == 0 or index == len(CASES):
        print(f"Processed {index}/{len(CASES)} cases")

RUN_DURATION_SECONDS = round(time.perf_counter() - RUN_STARTED, 3)
TRACE.emit(
    None,
    "CoordinatorAgent",
    "run_completed",
    {
        "cases_processed": len(RESULTS),
        "duration_seconds": RUN_DURATION_SECONDS,
        "qwen_validated_cases": sum(
            info["decision_source"] == "qwen_validated" for info in RUN_INFO.values()
        ),
        "deterministic_fallback_cases": sum(
            info["decision_source"] == "deterministic_fallback"
            for info in RUN_INFO.values()
        ),
    },
)


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


def hardware_metadata():
    result = {"gpu_count": 0, "gpus": []}
    try:
        import torch

        result["torch_version"] = torch.__version__
        result["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            result["gpu_count"] = torch.cuda.device_count()
            result["gpus"] = [
                {
                    "name": torch.cuda.get_device_name(index),
                    "memory_gib": round(
                        torch.cuda.get_device_properties(index).total_memory / 1024**3, 2
                    ),
                }
                for index in range(torch.cuda.device_count())
            ]
    except Exception as exc:
        result["torch_error"] = f"{type(exc).__name__}: {exc}"
    return result


qwen_validated_count = sum(
    info["decision_source"] == "qwen_validated" for info in RUN_INFO.values()
)
fallback_count = len(RUN_INFO) - qwen_validated_count
METADATA = {
    "model": MODEL_ID,
    "parameter_size": MODEL_PARAMETER_SIZE,
    "model_limit_compliance": "8.2B <= 10B per agent",
    "policy_version": POLICY_VERSION,
    "framework": {
        "orchestration": "custom Python structured-handoff multi-agent",
        "inference": "Hugging Face Transformers",
        "transformers_version": package_version("transformers"),
        "bitsandbytes_version": package_version("bitsandbytes"),
        "pandas_version": pd.__version__,
    },
    "runtime": {
        "environment": "kaggle" if IS_KAGGLE else "local",
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "started_at_utc": TRACE.started_at.isoformat(),
        "duration_seconds": RUN_DURATION_SECONDS,
        **hardware_metadata(),
    },
    "llm_backend": {
        "requested": ENABLE_LLM,
        "status": LLM.status,
        "source": LLM.model_source,
        "quantization": LLM.quantization,
        "generation": {
            "enable_thinking": False,
            "do_sample": False,
            "max_new_tokens": MAX_NEW_TOKENS,
        },
        "calls": LLM.calls,
        "parsed_responses": LLM.responses_parsed,
        "generation_errors": LLM.generation_errors,
        "load_errors": LLM.load_errors,
    },
    "run": {
        "run_id": TRACE.run_id,
        "cases_processed": len(RESULTS),
        "qwen_validated_cases": qwen_validated_count,
        "deterministic_fallback_cases": fallback_count,
        "verifier_repairs": sum(info["verifier_repaired"] for info in RUN_INFO.values()),
        "trace_events": len(TRACE.events),
    },
}

trace_paths = [ROOT_TRACE_PATH]
metadata_paths = [ROOT_METADATA_PATH]
if MIRROR_LOGGING:
    trace_paths.append(LOG_TRACE_PATH)
    metadata_paths.append(LOG_METADATA_PATH)
TRACE.flush(trace_paths)
metadata_text = json.dumps(METADATA, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
for metadata_path in dict.fromkeys(metadata_paths):
    atomic_write_text(metadata_path, metadata_text)

print(
    json.dumps(
        {
            "cases_processed": len(RESULTS),
            "duration_seconds": RUN_DURATION_SECONDS,
            "qwen_validated_cases": qwen_validated_count,
            "deterministic_fallback_cases": fallback_count,
            "output_dir": str(OUTPUT_DIR),
            "trace": str(ROOT_TRACE_PATH),
            "metadata": str(ROOT_METADATA_PATH),
        },
        indent=2,
    )
)


In [ ]:
# Hard-gate QA, golden aggregate assertions for the official 50-case set, and clean ZIP creation.
expected_output_names = {f"EC_{index:03d}.json" for index in range(1, EXPECTED_CASE_COUNT + 1)}
actual_output_paths = sorted(OUTPUT_DIR.glob("EC_*.json"))
actual_output_names = {path.name for path in actual_output_paths}
if actual_output_names != expected_output_names:
    raise AssertionError(
        f"Output filenames mismatch: missing={sorted(expected_output_names - actual_output_names)}, "
        f"extra={sorted(actual_output_names - expected_output_names)}"
    )

disk_results = {}
for path in actual_output_paths:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload["case_id"] != path.stem:
        raise AssertionError(f"case_id mismatch in {path.name}")
    if payload != RESULTS[path.stem]:
        raise AssertionError(f"Disk/in-memory mismatch in {path.name}")
    disk_results[path.stem] = payload

issue_counts = Counter(
    payload["assessment"]["primary_issue"] for payload in disk_results.values()
)
status_counts = Counter(
    payload["assessment"]["case_status"] for payload in disk_results.values()
)
aggregate_totals = {
    key: money_decimal(
        sum(
            (Decimal(str(payload["financial_resolution"][key])) for payload in disk_results.values()),
            Decimal("0"),
        )
    )
    for key in (
        "item_total_brl",
        "freight_total_brl",
        "payment_total_brl",
        "recommended_refund_brl",
    )
}

if STRICT_OFFICIAL_ASSERTIONS:
    expected_issue_counts = Counter(
        {
            "canceled_order_paid": 8,
            "unavailable_order_paid": 8,
            "late_delivery_seller": 8,
            "late_delivery_logistics": 8,
            "valid_split_payment": 9,
            "unsupported_late_claim": 9,
        }
    )
    expected_totals = {
        "item_total_brl": Decimal("4686.52"),
        "freight_total_brl": Decimal("727.47"),
        "payment_total_brl": Decimal("7782.89"),
        "recommended_refund_brl": Decimal("3429.64"),
    }
    assert issue_counts == expected_issue_counts, (issue_counts, expected_issue_counts)
    assert status_counts == Counter({"action_required": 32, "no_action": 18}), status_counts
    assert aggregate_totals == expected_totals, (aggregate_totals, expected_totals)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in actual_output_paths:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as archive:
    zip_names = archive.namelist()
    if len(zip_names) != EXPECTED_CASE_COUNT or set(zip_names) != expected_output_names:
        raise AssertionError("submission.zip must contain exactly the 50 EC_*.json basenames")
    if any("/" in name or "\\" in name for name in zip_names):
        raise AssertionError("JSON files must be at the ZIP root")

summary = pd.DataFrame(
    [{"primary_issue": issue, "cases": count} for issue, count in sorted(issue_counts.items())]
)
print(summary.to_string(index=False))
print("Status counts:", dict(status_counts))
print("Aggregate totals:", {key: str(value) for key, value in aggregate_totals.items()})
print(f"QA passed. Submission ready: {SUBMISSION_ZIP}")


## Kết quả chạy

Cell cuối chỉ hoàn tất khi:

- đủ đúng 50 output `EC_001.json` … `EC_050.json`;
- từng `case_id`, schema, entity/evidence limit, policy, party, action và tiền đã qua Verifier;
- tổng official khớp các golden assertions;
- `submission.zip` mở lại được và chỉ chứa 50 JSON ở ZIP root.

Kiểm tra `metadata.json` để biết lượt chạy đã dùng Qwen hay deterministic fallback. Việc fallback không bị che giấu: từng case có `decision_source` và `fallback_reason` trong `trace.jsonl`.
